In [ ]:
"""
IMPORT ALL IMPORTS, DEPENDENCIES, ETC NEEDED
"""
from huggingface_hub import login
from datasets import load_dataset
import unicodedata
import pandas as pd
import torch
from transformers import (AutoTokenizer,AutoModelForSeq2SeqLM,Trainer,TrainingArguments,)
from torch.utils.data import DataLoader
import editdistance

In [ ]:
"""
INITIAL DATASET LOAD FROM HUGGINGFACE

1. GET ACCESS TO YANKARI DATASET FROM https://huggingface.co/datasets/acflp/YANKARI
2. GENERATE A PERSONAL READ-ONLY TOKEN TO ACCESS IT
   2a. Go to your HuggingFace profile, under Settings/AccessTokens
   2b. Click '+ Create New Token'
   2c. Set Token Type to Read. Name Token. Create token. This is a 1-time display so save a copy.
   2d. Paste token in requested spot below.
"""

#INSERT YOUR HUGGINGFACE READ-ONLY TOKEN HERE INSIDE QUOTES
login("INSERT TOKEN HERE")

#LOAD THE DATASET, EXTRACT ONLY THE TEXT ENTRIES
ds = load_dataset("acflp/YANKARI", split="train")
ds_text = ds.remove_columns([col for col in ds.column_names if col != "text"])


"""
INITIAL DATA PREPROCESSING

THIS ENDS WITH PREPPED TRAINING DATASET SPLIT
PREPPED TRAINING ENTRIES INCLUDE:
1. Input text with no diacritic markings
2. Matching target text with diacritic markings

FORMAT:
{
    "input_text": "...",
    "target_text": "..."
}
"""
torch.cuda.empty_cache()
#RANDOMLY SPLIT 88% TRAIN
split_1 = ds_text.train_test_split(test_size=0.12, seed=42)
train_ds = split_1['train']
temp_ds = split_1['test']
#RANDOMLY SPLIT 6% DEV, 6% TEST
split_2 = temp_ds.train_test_split(test_size=0.5, seed=42)
dev_ds = split_2['train']
test_ds = split_2['test']
#ABLATE TRAIN TO 50%
split_3 = train_ds.train_test_split(test_size=0.5, seed=42)
train_ds = split_3['train']

In [ ]:
#DIACRITIC STRIPPING FUNCTION
#FOR CREATING DIACRITIC-FREE INPUT ENTRIES
def strip_diacritics(text):
    return ''.join(
        #USES UNICODE TO STANDARDIZE, SUCH AS ọ TO o
        c for c in unicodedata.normalize('NFD', text)
        if unicodedata.category(c) != 'Mn'
        )

#FUNCTION TO PREP INPUT
#GIVEN ENTRY, MAKES DIACRITIC-FREE INPUT, DIACRITIC MARKED TARGET
def preprocess_hf(example):
    return {
        "input_text": strip_diacritics(example["text"]),
        "target_text": example["text"]
    }

In [ ]:
#RUN TEXT PREPROCESSING ON ALL SPLITS
train_ds = train_ds.map(preprocess_hf, remove_columns=['text'],num_proc=6)
dev_ds = dev_ds.map(preprocess_hf, remove_columns=['text'],num_proc=6)
test_ds = test_ds.map(preprocess_hf, remove_columns=['text'],num_proc=6)

#USE TRAIN SET FOR TRAINING
dataset = train_ds

"""
MODEL SETUP

THIS ENDS WITH BYT5 MODEL LOADED, TOKENIZATION FUNCTIONS READY
"""

In [ ]:
#LOAD BYT5 MODEL AND TOKENIZER
#CURRENTLY USING SMALL, WE CAN TRY UPPING TO BYT5 BASE
model_name = "google/byt5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

#FREEZE ENCODER LAYERS
#CURRENTLY FREEZES HALF TO REDUCE TRAINING TIME, WE CAN TRY UNFREEZING MORE
num_encoder_layers = len(model.encoder.block)
for layer in model.encoder.block[:num_encoder_layers // 2]:
    for param in layer.parameters():
        param.requires_grad = False

#TOKENIZATION FUNCTION
def preprocess(example):
    #TOKENIZE INPUT
    model_inputs = tokenizer(
        example["input_text"],
        truncation=True,
        #PADS SHORT ENTRIES TO 256
        padding="max_length",
        #CAPS LONG ENTRIES TO 256
        #WE MIGHT WANT TO EXTEND THIS, OR IMPLEMENT SLIDING WINDOW, SINCE THIS LOSES INPUT INFORMATION
        max_length=256,
    )
    #TOKENIZE TARGET, SAME SETUP
    labels = tokenizer(
        example["target_text"],
        truncation=True,
        padding="max_length",
        max_length=256,
    )["input_ids"]

    #PADDING TOKENS
    #LIST COMPREHENSION FOR ALL LABEL IDS, WITH ALL PADDING TOKENS SET TO -100
    #CROSS ENTROPY LOSS IGNORES VALUE -100
    labels = [(l if l != tokenizer.pad_token_id else -100) for l in labels]
    model_inputs["labels"] = labels
    return model_inputs

"""
RUN TOKENIZATION OF TRAINING SPLIT

ENDS WITH TOKENIZED TRAINING SPLIT
"""

#GET TOKENIZED DATASET
tokenized_dataset = dataset.map(preprocess)

#FORMAT TOGETHER WITH INPUTS, ATTENTION MASK, LABELS
tokenized_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

In [ ]:
"""
DEFINE TRAINER CLASS, SET TRAINING HYPERPARAMETERS

ENDS WITH MODEL READY TO TRAIN
"""

#TRAINER CLASS
class YorubaTrainer(Trainer):
    #INIT
    def __init__(self, byte_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.byte_weights = byte_weights

    #FUNCTION TO GET LOSS
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        outputs = model(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            labels=inputs["labels"],
        )
        loss = outputs.loss
        return (loss, outputs) if return_outputs else loss

#TRAINING VARIABLES
training_args = TrainingArguments(
    learning_rate=1e-4,
    #TRAIN BATCH SIZE, MAKE SURE SAME AS EVAL
    gradient_accumulation_steps=16,
    per_device_train_batch_size=1,
    num_train_epochs=15,
    weight_decay=0.01,
    #FOR PROGRESS MONITORING - PRINTS LOSS EVERY N STEPS DURING TRAINING
    logging_steps=50,
    #CURRENTLY NO INTERMITENT SAVING, WAS BREAKING
    save_strategy="no",
    report_to="none",
    bf16=True,
    gradient_checkpointing=True,
    disable_tqdm=False,
    logging_strategy='steps'
)

#ACTUAL TRAINER
trainer = YorubaTrainer(
    #PASSES IN ALL OF ABOVE
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)

In [ ]:
print(max(len(x) for x in tokenized_dataset["input_ids"]))
print(max(len(x) for x in tokenized_dataset["labels"]))
print(sum(len(x) for x in tokenized_dataset["input_ids"]) / len(tokenized_dataset))

In [ ]:
import os
import gc

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:64"

gc.collect()
torch.cuda.empty_cache()

"""
RUN MODEL TRAINING
"""

#RUN THE TRAINER
trainer.train()

"""
SAVE MOVEL PARAMETERS TO LOCAL MAIN DIRECTORY
"""
#CHANGE PATHS INSIDE QUOTES TO CHANGE SAVE DIRECTORY
model.save_pretrained("./byt5_yoruba_50")
tokenizer.save_pretrained("./byt5_yoruba_50")
